# Vintage workbook delta comparison

Compares two structurally identical vintage workbooks and writes a third that **mirrors their layout
with the deltas as the cell values** — same sheets, same header blocks, same category columns, and
`B − A` everywhere else.

The engine lives in `vintage_compare.py`, which must sit next to this notebook.

## The layout as specified

| | `Vintage Summary` | `Vintage1` … `Vintage24` |
|---|---|---|
| Copied through as-is | rows 1–24 (the charts block) | rows 1–9 |
| Compared block | `A25:Y756` | `A10:AA248` |
| Category columns (kept from A, checked for alignment) | `A` | `A` and `B` |
| Differences ignored entirely | — | column `A` |
| Delta columns | `B`–`Y` | `C`–`AA` |
| Row alignment | 1:1 | 1:1 to row 200, then **row 201 of the *snow* workbook is skipped** and base 201 ↔ snow 202 … base 248 ↔ snow 249 |

Values only. Formulas are compared on their cached results; charts, images and formatting are ignored.

## Assumptions I had to make — check these first

1. **24 vintage sheets** (`Vintage Summary` + `Vintage1…24` = 25). Your message said "vintage1 to
   vintage24" in one place and "vintage 1 to 26" in another. Change `N_VINTAGES` below if it's 26.
2. **Delta direction is `snow − non-snow`.** The snow workbook is the one carrying the extra row, so
   it reads as the candidate. Swap `BASE_FILE` and `OTHER_FILE` to flip the sign.
3. **Row 248 is the *base* workbook's last row**, so the snow workbook runs to 249. Section 4 checks
   this against the real files.
4. **The summary sheet has no row shift** — the +1 was described only for the vintage sheets. If the
   snow summary also runs one row long, section 4 will show it and you add the same `insertions` rule.
5. **A blank on one side is treated as zero**, so base `100` vs blank gives `−100`. Every such cell is
   listed on the `_Issues` tab, so nothing is silent. Set `blank_as_zero=False` to leave them empty.
6. **Text in the delta region**: identical text passes through unchanged so the sheet still reads
   normally; differing text is written as `old -> new` and flagged.

## Output

| Sheet | Contents |
|---|---|
| `_Summary` | Per sheet: rows compared, row shift applied, which snow rows were skipped, cells compared, non-zero deltas, largest movement, category mismatches |
| `_Issues` | Every category mismatch, blank-on-one-side and type clash, with both values and both row numbers |
| `Vintage Summary`, `Vintage1`… | The mirrored sheets. Non-zero deltas are highlighted by a conditional-format rule |

Run section 4 (inspect) **before** trusting section 5 — it checks the declared layout against what the
files actually contain.

## 1. Imports
`vintage_compare.py` must be in the same folder as this notebook. Re-run this cell after editing it.

In [ ]:
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import vintage_compare as vc
importlib.reload(vc)                 # so edits to the module land without restarting the kernel

try:
    import pandas as pd
except ImportError:
    pd = None

print("vintage_compare loaded from", Path(vc.__file__).resolve())
print("readers: .xlsx/.xlsm (openpyxl), .xlsb (pyxlsb), .xls (xlrd)")

## 2. Settings
Set the two paths. Everything else already matches the layout described above.

In [ ]:
# ============================================================================
#  EDIT THIS CELL
# ============================================================================

BASE_FILE   = "vintage_report.xlsx"        # A - the workbook WITHOUT "snow" in its name
OTHER_FILE  = "vintage_report_snow.xlsx"   # B - the workbook WITH "snow" (one row longer)
OUTPUT_FILE = "vintage_deltas.xlsx"        # written here, overwritten if it exists

N_VINTAGES    = 24                         # Vintage1 .. VintageN
SUMMARY_SHEET = "Vintage Summary"          # matched loosely: "vintage summary", "VintageSummary", ...
VINTAGE_NAME  = "Vintage{}"                # matched loosely: "Vintage 1", "VINTAGE_01", ...

OPT = vc.DeltaOptions(
    blank_as_zero=True,          # base 100 vs blank -> -100, and flagged on _Issues
    keep_matching_text=True,     # identical text passes through, so the sheet still reads normally
    abs_tolerance=1e-9,          # movements smaller than this are written as 0
    category_similarity=0.85,    # below this, two category names are called a mismatch
    zero_as_blank=False,         # True writes blanks instead of 0 where nothing moved
)

SPECS = vc.vintage_specs(n_vintages=N_VINTAGES,
                         summary_name=SUMMARY_SHEET,
                         vintage_name=VINTAGE_NAME)

# Sanity check on which file is which - the snow workbook is the one with the extra row.
if "snow" in Path(BASE_FILE).name.lower():
    print("WARNING: BASE_FILE looks like the snow workbook. The +1 row rule is applied to "
          "OTHER_FILE, so these are probably the wrong way round.")

print(f"{len(SPECS)} sheets declared: {SPECS[0].name} + {SPECS[1].name}..{SPECS[-1].name}")
for s in (SPECS[0], SPECS[1]):
    print(f"  {s.name:<18} block {s.first_col}{s.first_row}:{s.last_col}{s.last_row}"
          f"  headers {s.header_rows}  keys {s.key_cols}  ignored {s.ignore_cols}"
          f"  insertions {s.insertions}")

## 3. The row alignment, spelled out
Prints exactly which base row is compared against which snow row around the row-201 boundary, and which
snow rows are dropped. Read this before running the comparison — if it doesn't match what you expect,
nothing downstream will be right.

In [ ]:
for spec in (SPECS[0], SPECS[1]):
    row_map, skipped = vc.build_row_map(spec)
    print(f"\n{spec.name}   ({len(row_map)} rows compared)")
    print(f"  block {spec.first_col}{spec.first_row}:{spec.last_col}{spec.last_row}")
    if not spec.insertions:
        print(f"  base {spec.first_row}-{spec.last_row}  ->  other {spec.first_row}-{spec.last_row}"
              "   (1:1, no shift)")
        continue
    boundary = spec.insertions[0][0]
    around = [r for r in (boundary - 1, boundary, boundary + 1, boundary + 2, spec.last_row)]
    for r in around:
        print(f"  base row {r:>4}  ->  other row {row_map[r]:>4}")
    print(f"  skipped in the other workbook: {skipped}")

## 4. Inspect the real files before comparing
Checks the declared layout against what the workbooks actually contain: does every sheet resolve, where
does the data really start and stop, and does the snow workbook genuinely run one row longer.

**What you want to see:** every sheet found, `Data past last_row` = 0 for the base workbook and 1 for
the snow vintage sheets. If the snow *summary* also shows 1, it has the same inserted row and you should
add `insertions=((200, 1),)` to the summary spec.

In [ ]:
for label, path in (("BASE (A)", BASE_FILE), ("OTHER/snow (B)", OTHER_FILE)):
    print(f"\n=== {label}: {path}")
    if not Path(path).exists():
        print("   NOT FOUND - set the path in section 2")
        continue
    rows = vc.inspect_workbook(path, SPECS)
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for r in rows:
            print("  ", r)

    missing = [r["Spec sheet"] for r in rows if r["Found as"] == "NOT FOUND"]
    if missing:
        print(f"   {len(missing)} sheet(s) not found: {missing}")
        print("   -> check SUMMARY_SHEET / VINTAGE_NAME / N_VINTAGES in section 2")

## 5. Run the comparison and write the delta workbook

In [ ]:
if not (Path(BASE_FILE).exists() and Path(OTHER_FILE).exists()):
    result = None
    print("Set BASE_FILE and OTHER_FILE in section 2 first.")
    print("Section 7 runs the whole thing on generated workbooks if you want to see it work.")
else:
    result = vc.compare_workbooks(BASE_FILE, OTHER_FILE, SPECS, OPT)
    out_path = vc.write_delta_workbook(result, OUTPUT_FILE)
    vc.print_summary(result, out_path)

## 6. Explore the result
Biggest movements and every category that didn't line up. Check the mismatches before circulating the
deltas — a mismatched category means the two sheets disagree about what that row *is*, which makes the
delta on that row meaningless.

In [ ]:
if result is None:
    print("Run section 5 first.")
elif pd is None:
    print("pandas is not installed - `pip install pandas` to use this section.")
else:
    issues = pd.DataFrame([i for r in result["results"] for i in r.issues],
                          columns=vc.ISSUE_COLUMNS)
    cats = issues[issues["Issue"].str.startswith("category")] if not issues.empty else issues
    print(f"{len(cats)} category problems, {len(issues) - len(cats)} blank/type problems\n")
    if not cats.empty:
        print("Category mismatches - the rows where the two workbooks disagree about the label")
        display(cats[["Sheet", "Cell", "Base value", "Other value", "Issue"]].head(40))

    moved = []
    for r in result["results"]:
        for (row, col), v in r.values.items():
            if vc.is_number(v) and v != 0 and col not in set(r.spec.key_col_indices()):
                moved.append({
                    "Sheet": r.base_sheet,
                    "Cell": f"{vc.get_column_letter(col)}{row}",
                    "Category": r.values.get((row, r.spec.key_col_indices()[-1])),
                    "Delta": v,
                })
    moved_df = pd.DataFrame(moved)
    if not moved_df.empty:
        print(f"\n{len(moved_df):,} cells moved. Largest movements:")
        top = moved_df.reindex(moved_df["Delta"].abs().sort_values(ascending=False).index)
        display(top.head(25))
        print("\nBy sheet:")
        display(moved_df.groupby("Sheet")["Delta"]
                .agg(cells="count", total="sum", largest=lambda s: s.abs().max()))
    else:
        print("\nNo cell moved.")

## 7. Self-test on generated workbooks
Builds a base workbook and a "snow" workbook one row longer, with known differences planted at the
awkward places — right at the row-201 boundary, on the last row, a renamed category, a changed column A
(which must be ignored), and a blank on one side — then asserts the engine finds exactly those and
nothing else. Run it to confirm the notebook works before pointing it at the real files.

In [ ]:
RUN_SELF_TEST = True      # set to False once you are pointing at the real workbooks

if RUN_SELF_TEST:
    import tempfile
    from openpyxl import Workbook, load_workbook

    demo_dir = Path(tempfile.mkdtemp(prefix="vintage_selftest_"))
    N_DEMO, SUMMARY_LAST, VINT_LAST = 3, 756, 248

    def build_demo(path, snow: bool):
        """The snow workbook gets one extra row after row 200 on every vintage sheet."""
        wb = Workbook()
        wb.remove(wb.active)

        ws = wb.create_sheet("Vintage Summary")
        for r in range(1, 25):
            ws.cell(row=r, column=1, value=f"chart block line {r}")
        for r in range(25, SUMMARY_LAST + 1):
            ws.cell(row=r, column=1, value=f"Category {r}")
            for c in range(2, 26):                                   # B..Y
                ws.cell(row=r, column=c,
                        value=r * 10 + c + (5 if snow and r == 100 and c == 3 else 0))

        for v in range(1, N_DEMO + 1):
            ws = wb.create_sheet(f"Vintage {v}")                     # a space, to exercise matching
            for r in range(1, 10):
                ws.cell(row=r, column=1, value=f"header {r}")
            for base_row in range(10, VINT_LAST + 1):
                out_row = base_row + 1 if (snow and base_row > 200) else base_row
                ws.cell(row=out_row, column=1,
                        value=f"A{base_row}" + ("_CHANGED" if snow and base_row == 50 else ""))
                label = f"Category {base_row}"
                if snow and base_row == 60:
                    label = "Something Completely Different"          # must be flagged
                if base_row == 61:
                    label = "Salaries and Wages" if snow else "Salaries & Wages"   # must NOT flag
                ws.cell(row=out_row, column=2, value=label)
                for c in range(3, 28):                               # C..AA
                    val = base_row * 100 + c
                    if snow and base_row == 201 and c == 3:
                        val += 50                                    # just past the inserted row
                    if snow and base_row == 248 and c == 27:
                        val -= 7                                     # the very last cell
                    if base_row == 70 and c == 4:
                        if snow:
                            continue                                 # blank on one side
                        val = 1234
                    ws.cell(row=out_row, column=c, value=val)
            if snow:                                                 # the row that must be skipped
                ws.cell(row=201, column=1, value="INSERTED_A")
                ws.cell(row=201, column=2, value="INSERTED ROW - MUST NOT APPEAR")
                for c in range(3, 28):
                    ws.cell(row=201, column=c, value=-999_999)
        wb.save(path)

    demo_base, demo_snow = demo_dir / "demo_base.xlsx", demo_dir / "demo_snow.xlsx"
    build_demo(demo_base, snow=False)
    build_demo(demo_snow, snow=True)

    demo_specs = vc.vintage_specs(n_vintages=N_DEMO)
    demo_result = vc.compare_workbooks(demo_base, demo_snow, demo_specs, OPT)
    demo_out = vc.write_delta_workbook(demo_result, demo_dir / "demo_deltas.xlsx")
    vc.print_summary(demo_result, demo_out)

    # --- row alignment ------------------------------------------------------
    row_map, skipped = vc.build_row_map(demo_specs[1])
    assert row_map[200] == 200 and row_map[201] == 202 and row_map[248] == 249
    assert skipped == [201] and len(row_map) == 239
    assert vc.build_row_map(demo_specs[0])[1] == [], "the summary sheet takes no row shift"

    summary_res, v1 = demo_result["results"][0], demo_result["results"][1]

    # --- the summary sheet --------------------------------------------------
    assert summary_res.values[(100, 3)] == 5
    assert summary_res.nonzero_deltas == 1
    assert summary_res.values[(1, 1)] == "chart block line 1", "rows 1-24 pass through"
    assert summary_res.values[(25, 1)] == "Category 25", "column A comes from the base workbook"
    assert (25, 26) not in summary_res.values, "nothing past column Y"

    # --- the vintage sheets, across the inserted row ------------------------
    assert v1.values[(200, 3)] == 0, "base 200 vs snow 200"
    assert v1.values[(201, 3)] == 50, "base 201 vs snow 202"
    assert v1.values[(248, 27)] == -7, "base 248 vs snow 249, the last cell"
    assert not any(isinstance(x, str) and "INSERTED" in x for x in v1.values.values()), \
        "the skipped snow row leaked into the output"
    assert -999_999 not in [x for x in v1.values.values() if vc.is_number(x)], \
        "the skipped snow row leaked into the output"
    assert v1.values[(1, 1)] == "header 1", "rows 1-9 pass through"

    # --- categories ---------------------------------------------------------
    flagged = {(i["Cell"], i["Issue"].split(" (")[0]) for i in v1.issues}
    assert not any(i["Column"] == "A" for i in v1.issues), "column A differences must be ignored"
    assert v1.values[(50, 1)] == "A50", "column A still comes through from the base workbook"
    assert ("B60", "category mismatch") in flagged, sorted(flagged)
    assert not any(c == "B61" for c, _ in flagged), "'&' vs 'and' is the same category"
    assert any(c == "D70" for c, _ in flagged), "a blank on one side must be flagged"
    assert v1.values[(70, 4)] == -1234, "blank_as_zero -> a full -1234 movement"
    assert v1.category_mismatches == 1 and v1.nonzero_deltas == 3

    # --- the written workbook -----------------------------------------------
    wb = load_workbook(demo_out)
    ws = wb["Vintage 1"]
    assert ws["C201"].value == 50 and ws["AA248"].value == -7
    assert ws["B60"].value == "Category 60" and ws["D70"].value == -1234
    assert ws.max_row == VINT_LAST and ws.max_column == 27
    assert wb["Vintage Summary"].max_row == SUMMARY_LAST
    assert wb["Vintage Summary"].max_column == 25
    assert wb["_Issues"].max_row - 1 == N_DEMO * 2

    print("\nself-test")
    print("  row 201 skipped, 201->202 and 248->249 aligned, categories checked, column A ignored")
    print(f"  demo files: {demo_dir}")